***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.3 傅里叶级数：从周期展开到谱表示](2_3_fourier_series.ipynb)
    * 下一节：[2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.4 连续傅里叶变换：从信号表示到可见度语言<a id='math:sec:the_fourier_transform'></a>

傅里叶级数处理周期函数，它的频率只出现在基频的整数倍上。连续傅里叶变换处理非周期或局域函数，频率变量变成连续变量。这个变化对射电干涉测量非常关键：天空亮度分布不是一个人为周期函数，而干涉阵在不同基线上测到的是连续空间频率平面上的样本。

本节的重点有三层。第一，傅里叶变换可以看成“把函数投影到连续复指数基底上”；第二，原始域与傅里叶域存在尺度互反、位移变相位、尖锐结构产生宽频响应等基本直觉；第三，在小视场近似下，天空亮度 $I(l,m)$ 与可见度 $V(u,v)$ 的关系正是一组二维傅里叶变换。完整测量方程会在后续章节加入主波束、$w$ 项、极化和校准项，这里先建立核心数学语言。


### 2.4.1 从级数到变换<a id='math:sec:from_series_to_transform_definition'></a>

第 2.3 节的复系数傅里叶级数为

$$
f_T(t)=\sum_{m=-\infty}^{+\infty}a_m e^{i2\pi mt/T},
\qquad
a_m=\frac{1}{T}\int_{-T/2}^{T/2}f_T(t)e^{-i2\pi mt/T}\,dt.
$$

令离散频率 $s_m=m/T$，频率间隔为 $\Delta s=1/T$。当周期 $T$ 增大时，$s_m$ 之间的间隔变小；若函数主要集中在有限范围内，周期边界被推到很远处，离散求和就逐渐过渡为连续频率积分。于是得到连续傅里叶变换对：

<a id='math:eq:continuous_ft_pair'></a><!--\label{math:eq:continuous_ft_pair}-->
$$
F(s)=\mathcal{F}\{f\}(s)=\int_{-\infty}^{+\infty}f(x)e^{-2\pi ixs}\,dx,
$$

<a id='math:eq:continuous_ift_pair'></a><!--\label{math:eq:continuous_ift_pair}-->
$$
f(x)=\mathcal{F}^{-1}\{F\}(x)=\int_{-\infty}^{+\infty}F(s)e^{+2\pi ixs}\,ds.
$$

这里 $x$ 与 $s$ 是一对共轭变量。它们可以表示时间与频率、角坐标与空间频率、孔径坐标与远场方向余弦，也可以表示后面最常用的天空坐标 $(l,m)$ 与 `uv` 坐标 $(u,v)$。若 $x$ 的单位是秒，$s$ 的单位就是 Hz；若 $x$ 是弧度，$s$ 就是“每弧度的周期数”。变量单位互为倒数，是理解尺度互反的第一步。


严格的傅里叶反演定理需要函数满足适当条件，例如绝对可积、平方可积或在分布意义下解释。本书后续主要使用物理上足够良好的函数、有限能量信号以及 delta 这类标准分布，因此不在这里展开泛函分析细节。需要记住的是：正变换用负号指数分析函数中的频率成分，逆变换用正号指数把这些成分重新叠加回来。

为了书写紧凑，常把傅里叶变换记为

$$
F(s)=\tilde f(s)=\mathcal{F}\{f\}(s),
\qquad
f\rightleftharpoons F.
$$

不同教材会把 $2\pi$ 放在指数外的归一化因子中，或者交换正负号。只要在一本书内保持一致，物理结论不变；但在比较软件和文献公式时，必须先确认采用的是哪一种约定。


### 2.4.2 傅里叶变换在“测量”什么<a id='math:sec:what_ft_measures'></a>

从定义看，$F(s)$ 是 $f(x)$ 与复指数 $e^{2\pi ixs}$ 的内积。给定一个频率 $s$，积分

$$
\int f(x)e^{-2\pi ixs}\,dx
$$

会把与该频率同相的贡献相干相加，把快速变号或相位不匹配的贡献相互抵消。因此 $F(s)$ 的模表示该频率分量有多强，辐角表示它相对选定原点的相位。

这个解释在干涉测量中极其直接。一条基线对应一个空间频率点 $(u,v)$；相关器输出的复可见度就是天空亮度在相应傅里叶核 $e^{-2\pi i(ul+vm)}$ 上的投影。换言之，一条基线不是“拍一张小图”，而是在测量天空亮度分布的一个傅里叶分量。


### 2.4.3 基本变换对：尺度、边界与相位<a id='math:sec:basic_ft_pairs'></a>

高斯、矩形窗和 delta 函数是最需要先掌握的三个变换对。它们分别体现三种常见直觉：平滑局域结构在两个域中都平滑，硬边界会产生 `sinc` 旁瓣，点源会产生常幅度的相位斜坡。

![基本傅里叶变换对](figures/fourier_transform_basic_pairs.png)

**图 2.4.1** 三组基本傅里叶变换对。高斯的傅里叶变换仍为高斯，矩形窗的傅里叶变换为 `sinc`，移位 delta 的傅里叶变换具有恒定幅度和线性相位。


#### 高斯函数

对

$$
f(x)=a\exp\left[-\frac{(x-\mu)^2}{2\sigma^2}\right],
$$

傅里叶变换为

<a id='math:eq:gaussian_ft'></a><!--\label{math:eq:gaussian_ft}-->
$$
\mathcal{F}\{f\}(s)
=a\sqrt{2\pi}\sigma\,
\exp(-2\pi^2\sigma^2s^2)\,
\exp(-2\pi i\mu s).
$$

这个结果包含两部分。若中心在原点，变换仍是高斯；若中心平移到 $\mu$，傅里叶域额外乘上相位因子 $e^{-2\pi i\mu s}$。对归一化高斯

$$
f(x)=\frac{1}{\sqrt{2\pi}\sigma}\exp\left(-\frac{x^2}{2\sigma^2}\right),
$$

有

<a id='math:eq:normalized_gaussian_ft'></a><!--\label{math:eq:normalized_gaussian_ft}-->
$$
F(s)=\exp(-2\pi^2\sigma^2s^2)
=\exp\left(-\frac{s^2}{2\sigma_s^2}\right),
\qquad
\sigma_s=\frac{1}{2\pi\sigma}.
$$

因此 $\sigma\sigma_s=1/(2\pi)$。原始域越窄，傅里叶域越宽；原始域越宽，傅里叶域越窄。这个尺度互反关系会出现在分辨率、带宽、时间平均和孔径大小的所有讨论中。


高斯变换的推导只需要完成平方。把 $x'=x-\mu$，则

$$
\begin{aligned}
F(s)
&=ae^{-2\pi i\mu s}\int_{-\infty}^{+\infty}
\exp\left[-\frac{{x'}^2}{2\sigma^2}-2\pi isx'\right]dx'\\
&=ae^{-2\pi i\mu s}\int_{-\infty}^{+\infty}
\exp\left[-\frac{1}{2\sigma^2}\left(x'+2\pi i\sigma^2s\right)^2
-2\pi^2\sigma^2s^2\right]dx'\\
&=a\sqrt{2\pi}\sigma e^{-2\pi^2\sigma^2s^2}e^{-2\pi i\mu s}.
\end{aligned}
$$

最后一步使用的是高斯积分。复平面中的平移路径不会改变该高斯积分的值；在本教材层面，可以把它理解为完成平方后仍然留下同一个高斯面积因子。


#### 矩形窗与 `sinc`

令宽度为 $W$ 的矩形窗为

$$
\Pi_W(x)=
\begin{cases}
1, & |x|\le W/2,\\
0, & |x|>W/2.
\end{cases}
$$

直接积分得到

<a id='math:eq:rectangle_ft'></a><!--\label{math:eq:rectangle_ft}-->
$$
\begin{aligned}
\mathcal{F}\{\Pi_W\}(s)
&=\int_{-W/2}^{W/2}e^{-2\pi ixs}\,dx\\
&=W\frac{\sin(\pi Ws)}{\pi Ws}
=W\operatorname{sinc}(Ws).
\end{aligned}
$$

这说明有限范围会带来振荡旁瓣。矩形窗越宽，`sinc` 主瓣越窄；矩形窗越窄，傅里叶域响应越宽。有限孔径的衍射主瓣、有限通道宽度造成的带宽响应、有限积分时间造成的时间平均响应，都可以从这条变换对获得第一层直觉。


#### delta 函数与相位斜坡

delta 函数的抽样性质立即给出

<a id='math:eq:delta_ft'></a><!--\label{math:eq:delta_ft}-->
$$
\mathcal{F}\{\delta(x-x_0)\}(s)
=\int\delta(x-x_0)e^{-2\pi ixs}\,dx
=e^{-2\pi ix_0s}.
$$

当 $x_0=0$ 时，$\delta(x)$ 的傅里叶变换是常数 1。移位只改变相位，不改变幅度。对干涉测量而言，这正是点源可见度的基本形状：理想点源在所有空间频率上都有相干响应，而它偏离相位中心的位置表现为随 $(u,v)$ 线性变化的相位。

常数函数与 delta 函数互为变换对：

$$
1\rightleftharpoons \delta(s),
\qquad
\delta(x)\rightleftharpoons 1.
$$

这也解释了为什么一个完全均匀、无限延展的结构只在零空间频率处有响应。


#### Shah 函数

用第 2.2 节的约定

$$
\operatorname{III}_T(x)=\sum_{n=-\infty}^{+\infty}\delta(x-nT),
$$

它的傅里叶变换为

<a id='math:eq:shah_ft'></a><!--\label{math:eq:shah_ft}-->
$$
\mathcal{F}\{\operatorname{III}_T(x)\}(s)
=\frac{1}{T}\operatorname{III}_{1/T}(s).
$$

这个关系是采样理论的核心。原始域的规则采样会在傅里叶域中产生周期复制，复制间隔为 $1/T$。采样越稀疏，复制谱越密；一旦复制谱彼此重叠，就会发生混叠。


### 2.4.4 对称性：实函数、偶函数与共轭关系<a id='math:sec:ft_symmetry'></a>

傅里叶变换把实偶性、奇偶性和复共轭结构以非常规则的方式带到另一个域中。若 $f(x)$ 是实函数，则

<a id='math:eq:real_to_hermitian'></a><!--\label{math:eq:real_to_hermitian}-->
$$
F^*(s)=F(-s).
$$

这称为 Hermitian 对称性。证明只需对定义取复共轭：

$$
\begin{aligned}
F^*(s)
&=\left[\int f(x)e^{-2\pi ixs}\,dx\right]^*\\
&=\int f(x)e^{+2\pi ixs}\,dx
=\int f(x)e^{-2\pi ix(-s)}\,dx
=F(-s).
\end{aligned}
$$

若 $f(x)$ 同时是实偶函数，则 $F(s)$ 也是实偶函数；若 $f(x)$ 是实奇函数，则 $F(s)$ 是纯虚奇函数。这个性质会在可见度共轭对称中再次出现：若天空亮度是真实的实值分布，则理想可见度满足 $V(-u,-v)=V^*(u,v)$。


复共轭本身也有简单的变换关系：

<a id='math:eq:conjugate_ft'></a><!--\label{math:eq:conjugate_ft}-->
$$
\mathcal{F}\{f^*\}(s)=F^*(-s).
$$

此外，连续傅里叶变换连续做两次会得到反射函数：

<a id='math:eq:double_ft_reflection'></a><!--\label{math:eq:double_ft_reflection}-->
$$
\mathcal{F}\{\mathcal{F}\{f\}\}(x)=f(-x).
$$

这说明正变换和逆变换并不是两种完全陌生的运算；在本书采用的对称归一化约定下，它们只差指数符号，而两次同向变换等价于变量反号。


### 2.4.5 多维傅里叶变换与 `uv` 平面<a id='math:sec:multidimensional_ft_uv'></a>

多维傅里叶变换只是把乘积 $xs$ 换成点积 $\mathbf{x}\cdot\mathbf{s}$。对 $n$ 维函数，有

<a id='math:eq:multidimensional_ft'></a><!--\label{math:eq:multidimensional_ft}-->
$$
F(\mathbf{s})=
\int_{\mathbb{R}^n}f(\mathbf{x})
\exp[-2\pi i(\mathbf{x}\cdot\mathbf{s})]d^n\mathbf{x},
$$

$$
f(\mathbf{x})=
\int_{\mathbb{R}^n}F(\mathbf{s})
\exp[+2\pi i(\mathbf{x}\cdot\mathbf{s})]d^n\mathbf{s}.
$$

在小视场、窄带并暂时忽略主波束和 $w$ 项的近似下，射电干涉测量最核心的形式可写为

<a id='math:eq:visibility_fourier_core'></a><!--\label{math:eq:visibility_fourier_core}-->
$$
V(u,v)\approx\int\!\!\int I(l,m)
\exp[-2\pi i(ul+vm)]\,dl\,dm.
$$

这里 $(l,m)$ 是相对相位中心的方向余弦，$(u,v)$ 是以波长为单位的基线投影。这个式子暂时省略了许多真实效应，但它揭示了干涉成像的基本事实：每条基线测量天空亮度的一个二维傅里叶分量。

![二维天空分布与 uv 平面](figures/fourier_transform_sky_uv_pair.png)

**图 2.4.2** 一个简单二维天空模型及其傅里叶域幅度和相位。局域结构在 `uv` 平面中产生扩展响应；源的位置偏移会体现在相位结构中。


### 2.4.6 本节要点

连续傅里叶变换把函数写成连续复指数的叠加。它保留了第 2.3 节傅里叶级数的投影思想，但把离散谐波推广为连续频率。高斯展示了尺度互反，矩形窗展示了有限范围导致的 `sinc` 旁瓣，delta 函数展示了点源与相位斜坡，Shah 函数展示了采样与频谱复制。对射电干涉测量而言，这些不是抽象例子，而是后续可见度方程、采样理论、脏波束和成像反演的共同底层语言。


***

* 下一节：[2.5 卷积：响应函数如何重塑信号](2_5_convolution.ipynb)
